# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a guided walkthrough for loading and exploring the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

[https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json)

This open survey dataset contains mental health indicators, demographic data, and assessment scores (GAD-7, PHQ-9, PSQ) from Kilifi County, Kenya.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)

# Print metadata overview
md = dataset.metadata
print(f"{md.name}: {md.description}")
print(f"Published: {md.datePublished}")
print(f"Identifier: {md.identifier}")
print("Keywords:", md.keywords)
print("License:", md.license)
print("Fields with potential sensitive info:", md.personalSensitiveInformation)
print("Data Collection Summary:", md.dataCollection)


## 2. Data Overview
Review available record sets, fields, and their IDs. All references use entity `@id` per Croissant schema.

In [ ]:
# Inspect available record sets and their fields by @id
record_sets = dataset.record_sets
print("Available record sets (@id):")
for rs in record_sets:
    print(f"- {rs['@id']}: {rs.get('name', '<no name>')}")

# Let's choose the main record set: We'll assume it contains survey responses (most Croissant datasets have one named 'SurveyResponses' or similar)
# If multiple, list all, but for example, let's grab the first one.

if record_sets:
    main_record_set_id = record_sets[0]['@id']
    print(f"\nFields in record set {main_record_set_id}:")
    fields = dataset.fields(record_set=main_record_set_id)
    for fld in fields:
        print(f"- {fld['@id']} (name: {fld.get('name', '<no name>')}, dataType: {fld.get('dataType', None)})")
else:
    print("No record sets found.")

## 3. Data Extraction
Load data from available record sets into DataFrames for analysis.
Entities referenced by `@id` as required.

In [ ]:
# Extract all data from all record sets
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

# Display columns from the main record set
print(f"Columns in record set {main_record_set_id}:")
print(dataframes[main_record_set_id].columns.tolist())
print("\nFirst 5 records:")
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filter records, normalize numeric fields, categorize/group by attributes.
All field names reference `@id`.

In [ ]:
# Example EDA on a numeric mental health score field
# We'll look for GAD-7 score (Generalized Anxiety Disorder, 7-item scale)

# Find field @id for GAD-7
numeric_field_id = None
for fld in dataset.fields(record_set=main_record_set_id):
    name = fld.get('name', '').lower()
    if 'gad' in name or 'gad-7' in name:
        numeric_field_id = fld['@id']
        break
if numeric_field_id is None:
    # Try PHQ-9
    for fld in dataset.fields(record_set=main_record_set_id):
        name = fld.get('name', '').lower()
        if 'phq' in name or 'phq-9' in name:
            numeric_field_id = fld['@id']
            break

if numeric_field_id and numeric_field_id in dataframes[main_record_set_id].columns:
    # Set threshold for high anxiety/depression
    threshold = 10  # Standard cut-off for clinical significance
    filtered_df = dataframes[main_record_set_id][dataframes[main_record_set_id][numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize (z-score)
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try grouping by education level (find its @id)
    group_field_id = None
    for fld in dataset.fields(record_set=main_record_set_id):
        name = fld.get('name', '').lower()
        if 'education' in name:
            group_field_id = fld['@id']
            break

    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped mean {numeric_field_id} by {group_field_id}:")
        print(grouped_df.head())
    else:
        print("Education field not found for grouping.")
else:
    print("Could not find numeric field (GAD-7 or PHQ-9) in columns.")

## 5. Visualization
Visualize the distribution of the selected score (e.g., GAD-7 or PHQ-9) and its relation to education level.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Only run if numeric_field_id was found
if numeric_field_id and numeric_field_id in dataframes[main_record_set_id].columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(dataframes[main_record_set_id][numeric_field_id].dropna(), bins=10, kde=True)
    plt.xlabel(numeric_field_id)
    plt.title(f"Distribution of {numeric_field_id} scores")
    plt.show()

    if group_field_id and group_field_id in dataframes[main_record_set_id].columns:
        plt.figure(figsize=(10, 6))
        sns.boxplot(x=dataframes[main_record_set_id][group_field_id], y=dataframes[main_record_set_id][numeric_field_id])
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we've demonstrated how to load, overview, extract, and analyze the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using `mlcroissant`. 

**Key takeaways:**
- Data was accessed using Croissant entity `@id`s for maximum reproducibility and schema validity.
- Numeric fields (e.g., GAD-7, PHQ-9) can be filtered and normalized to examine clinical cutoffs.
- Group-level analysis (e.g., by education) surfaces potential disparities.
- Visualizations highlight score distributions and relationships across demographic dimensions.

This approach enables scalable, standards-compliant analysis of survey datasets for research and public health.